# Run the submission notebooks on Colab

Runs notebooks 01 to 05 of `submission/Misclassified_A03_EN3150/` one after another, on this
GPU runtime, so they share their files. Then pushes the executed notebooks and their results
to your fork. Notebooks 06 and 07 run on your laptop afterwards.

1. **Runtime > Change runtime type > T4 GPU > Save.**
2. **Runtime > Run all.** Allow access to your Colab Secrets if asked.
3. Keep the tab open. Training takes about 40 minutes; each epoch prints a line below.

Uses the same Colab Secrets as the section notebooks: `A03_REPO`, `A03_BRANCH`, `GH_TOKEN`,
`GIT_NAME`, `GIT_EMAIL`.

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# The commit this run uses, and the GPU it runs on (it should be a Tesla T4).
!git log --oneline -1
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip install --quiet nbclient nbformat

In [ ]:
# Run notebooks 01 to 05 in place; their printed output appears here as they run.
import subprocess

process = subprocess.Popen(
    [sys.executable, "tools/run_submission.py", "01", "02", "03", "04", "05"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError("a notebook failed (see above): nothing is pushed")

In [ ]:
# Send the executed notebooks and their results back to GitHub (data and weights stay here).
from edgecnn.utils import push_results

push_results(["submission/Misclassified_A03_EN3150"],
             "Run submission notebooks 01 to 05 (Colab)")